# Governance Dashboard — QuickSight Analysis for Drift Monitoring

This notebook creates or refreshes the Lab 5E QuickSight dashboard. The implementation lives in `src/governance/create_governance_dashboard.py`; the notebook stays a short driver.

The feature sheet compares normalized **drift magnitude**. A magnitude of `1.0` is the feature's method threshold, and values strictly above `1.0` are drifted. The detail table retains raw score, method, threshold, and direction for audit. Dataset-level `drifted_columns_share` remains the fraction of features that drifted.

Inference rows and drift runs join by `monitoring_run_id`, which Lab 5D stamps at scoring time. The join does not infer run membership from overlapping timestamps.

## What the build creates

The module creates one Athena data source, five direct-query datasets, a verified `feature_drift_detail` Athena view, a three-sheet analysis, and a published dashboard. It reconciles every discovered QuickSight user onto new and existing resources without revoking unrelated principals.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

import boto3

project_root = Path.cwd()
_env_dir = next((p for p in [project_root, *project_root.parents]
                 if (p / '.env').exists()), None)
if _env_dir is None:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')
load_dotenv(_env_dir / '.env', override=True)

_lab5_str = str(project_root)
if _lab5_str not in sys.path:
    sys.path.insert(0, _lab5_str)
for _k in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_k]

from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION
import src.governance.create_governance_dashboard as gov

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
ACCOUNT_ID = os.environ.get('ACCOUNT_ID') or \
    boto3.client('sts', region_name=AWS_DEFAULT_REGION).get_caller_identity()['Account']

print(f'✓ Loaded {_env_dir / ".env"}')
print(f'  PROJECT_NAME               = {PROJECT_NAME}')
print(f'  ACCOUNT_ID                 = {ACCOUNT_ID}')
print(f'  AWS_DEFAULT_REGION         = {AWS_DEFAULT_REGION}')
print(f'  QUICKSIGHT_IDENTITY_REGION = {QUICKSIGHT_IDENTITY_REGION}')

## 2. QuickSight prerequisite and data access

QuickSight **Enterprise** must be subscribed once per account. Subscription is billable and remains an explicit console or deployment choice. It is the only QuickSight setup step; you also need at least one QuickSight user, which a console sign-up creates for you.

**How QuickSight reads the data.** The IAM stack provisions a run-as role, `<ProjectName>-qs-datasource-role`, carrying two project-scoped policies (`<ProjectName>-QuickSightS3DataLakeAccess`, `<ProjectName>-QuickSightAthenaAccess`). The dashboard's Athena data source names that role (`AthenaParameters.RoleArn`), so QuickSight assumes it instead of the account-wide service role. The build therefore does not depend on the console-only *Quick access to AWS services* setting, and never modifies the shared `aws-quicksight-service-role-v0`. If the role is missing, the account was not provisioned with the current templates.

For automated provisioning, `SUBSCRIBE_QUICKSIGHT=true QUICKSIGHT_NOTIFICATION_EMAIL=you@example.com ./scripts/deploy-workshop.sh` subscribes before the IAM stack. The flag defaults to `false` so deployment never creates a billable subscription silently.

In [ ]:
quicksight_admin = boto3.client('quicksight', region_name=QUICKSIGHT_IDENTITY_REGION)
subscription = gov.check_quicksight_subscription(
    account_id=ACCOUNT_ID, quicksight_admin_client=quicksight_admin,
)

if subscription['subscribed']:
    print(f"✓ QuickSight active (Edition: {subscription['edition']})")
    if subscription['edition'] == 'STANDARD':
        print('  ⚠ Upgrade to Enterprise before building the dashboard.')
elif subscription['reason'] == 'access-denied':
    print('✗ This role cannot check QuickSight. Apply the Lab 5E IAM statement')
    print('  from templates/2-iam.yaml, then rerun this cell.')
else:
    print('✗ Subscribe this account to QuickSight Enterprise, then rerun this cell.')

## 3. Backfill trend history *(dev/test only)*

The trend visuals group `monitoring_timestamp` by **day**. Lab 5D runs all land on the same day, so without history every line chart shows a single point.

`src/governance/history_backfill.py` inserts one backdated `monitoring_responses` row per prior day (6 rows for `BACKFILL_DAYS = 7`), anchored on the latest real Lab 5D run for the endpoint:

- lineage (model package, snapshots, `model_version`) and baseline ROC-AUC are copied from that run;
- each feature's drift **magnitude** ramps from mild up to the real run's value, so the Feature Drift sheet trends as well;
- verdicts use the live rules: feature drift is magnitude `> 1.0`, dataset drift is `drifted_columns_share >= DATA_DRIFT_THRESHOLD`, and model drift is relative ROC-AUC loss `>= MODEL_DRIFT_THRESHOLD`.

Rows are tagged `monitoring_run_id = 'backfill-drift-day<N>-<date>'` and `detection_engine = 'evidently-backfill'`. They have no inference rows, so their inference counts are 0. The cell is idempotent: set `FORCE_BACKFILL = True` to rebuild after further Lab 5D runs. In a scheduled deployment each run appends its own row, so skip this section (`BACKFILL_HISTORY = False`).

In [ ]:
from src.governance.history_backfill import backfill_monitoring_history
import pandas as pd

BACKFILL_HISTORY = True   # False in a scheduled deployment with real history
BACKFILL_DAYS    = 7      # real run + 6 backdated days
FORCE_BACKFILL   = False  # True deletes existing backfill rows and rebuilds them

if BACKFILL_HISTORY:
    backfill = backfill_monitoring_history(days=BACKFILL_DAYS, force=FORCE_BACKFILL)
    if backfill['skipped']:
        print(f"✓ {backfill['existing']} backfill rows already present — skipped "
              f"(set FORCE_BACKFILL = True to rebuild).")
    else:
        print(f"✓ Inserted {backfill['inserted']} backfill rows anchored on "
              f"{backfill['anchor_run_id']} (deleted {backfill['deleted']}).")
    with pd.option_context('display.width', 160, 'display.max_colwidth', 48):
        print(backfill['summary'].to_string(index=False))
else:
    print('Backfill skipped — trend charts show only real monitoring runs.')

## 4. Build or refresh the dashboard

In [ ]:
from botocore.exceptions import ClientError

result = None
try:
    result = gov.create_dashboard(region=AWS_DEFAULT_REGION)
except gov.QuickSightUnavailable as exc:
    print(f'✗ Dashboard not created — {exc}')
except ClientError as exc:
    if exc.response['Error']['Code'] not in ('AccessDeniedException', 'UnrecognizedClientException'):
        raise
    print(f'✗ QuickSight denied {exc.operation_name}: {exc.response["Error"]["Message"]}')
    print('  Apply the full Lab 5E statement from templates/2-iam.yaml.')

if result:
    access = result['quicksight_service_role_permissions']
    if access['mode'] == 'run-as-role':
        print(f"✓ Data source runs as {access['role_arn']}")
        print('  (the account-wide QuickSight service role is not used or modified)')
    else:
        print('✓ QuickSight service-role preflight passed (stack has no run-as role)')
        print(f"  Policies attached by this run: {len(access['attached_now'])}")
    print(f"  Dashboard URL: {result['dashboard_url']}")
    print(f"  Analysis ARN:  {result['analysis_arn']}")

## Optional cleanup

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')